# Day 2, notebook 2: benchmark two vector stores the way production does

**Time: about 60 minutes. Work in pairs.** One person types, the other checks.

**What you will build.** A benchmark of the two vector stores in the syllabus, Azure AI Search and PostgreSQL with
pgvector, on the same data, scored against the same answer key. You measure what a search finds (recall), how fast
it is for one user, how it holds up when many users search at once, what a filter does to it, and what it costs per
million searches.

**Why it runs here.** In production you never time a store from your laptop. A laptop adds 200 to 300 ms of network
to every search, far more than the search itself, so every store looks the same. This notebook runs in Microsoft
Fabric in the same Azure region as both stores, Central India, the way an application server would. Step 0 measures the
trip so you can see it is small.

**How the cells work.** As in notebook 1: a cell that shows the real thing, a `YOUR CALL` cell you edit with your
prediction, and a cell that measures and prints your answer next to it.

**What your trainer set up before class.** The same made-up vectors, grouped in clusters the way real embeddings
are, loaded once into both stores: 100,000 vectors of 128 numbers, and 20,000 of 1,536 numbers (the size of
text-embedding-3-small). You only read. No model calls, so the only cost is the stores running.

In [ ]:
%pip install -q "psycopg[binary]" pgvector azure-search-documents matplotlib

In [ ]:
# Setup. Run this cell first. It fetches the lab code from the course repository and connects to both stores.
import io, json, os, pathlib, sys, time, urllib.request, zipfile
import numpy as np

REPO_ZIP = "https://github.com/mogupta84/ai-engineer-data-batch3/archive/refs/heads/main.zip"
TRAINER_COPY = pathlib.Path("/lakehouse/default/Files/batch3_repo.zip")      # trainer test runs only
ROOT = pathlib.Path("/tmp/freshcart")
data = TRAINER_COPY.read_bytes() if TRAINER_COPY.exists() else urllib.request.urlopen(REPO_ZIP, timeout=120).read()
with zipfile.ZipFile(io.BytesIO(data)) as z:
    for name in z.namelist():
        rel = name.split("/", 1)[1] if "/" in name else ""
        if rel.startswith(("src/", "lab2_ingest_index/")) and not name.endswith("/"):
            (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
            (ROOT / rel).write_bytes(z.read(name))
for folder in ("src", "lab2_ingest_index"):
    if str(ROOT / folder) not in sys.path:
        sys.path.insert(0, str(ROOT / folder))

# PASTE the three values your trainer gives you. Keep them in this notebook only.
SEARCH_ENDPOINT = "https://freshcart-bench-ci.search.windows.net"
SEARCH_QUERY_KEY = "PASTE-THE-QUERY-KEY"
PG_CONNECTION = "host=freshcart-pg-b3ci.postgres.database.azure.com port=5432 dbname=labs user=class_student password=PASTE-THE-PASSWORD sslmode=require"
settings = pathlib.Path("/lakehouse/default/Files/b3_settings.json")          # trainer test runs only
if settings.exists():
    s = json.loads(settings.read_text())
    SEARCH_ENDPOINT, SEARCH_QUERY_KEY, PG_CONNECTION = s["SEARCH_ENDPOINT"], s["SEARCH_QUERY_KEY"], s["PG_CONNECTION"]

from scale_set import SETS, make_benchmark_set, exact_top_k
from store_bench import (PgStore, SearchStore, recall, one_at_a_time, under_load, percentile, median)
pg = PgStore(PG_CONNECTION, SETS["128"]["table"])
ai = SearchStore(SEARCH_ENDPOINT, SEARCH_QUERY_KEY, SETS["128"]["index"])
print("pgvector table:", f"{pg.check(SETS['128']['n']):,} rows, graph index in place")
print("AI Search index:", f"{ai.check(SETS['128']['n']):,} rows")

## Step 0. How far away are the stores?

**What you will do.** Run the cell. It sends 50 trips to each store that do no search at all, and prints the middle
time.

**What you should see.** About 0.9 ms to PostgreSQL and 7.5 ms to AI Search for a trip with no
search in it. From a laptop the same trip took about 250 ms.

**Why this step is here.** Every number below includes this trip. If it were 250 ms, a 1 ms search and a 30 ms search
would look the same. Measure where the work runs before you trust any timing.

In [ ]:
import statistics
db = pg.db()
trip_pg, trip_ai = [], []
for _ in range(50):
    start = time.perf_counter()
    db.execute("SELECT 1").fetchall()                  # PostgreSQL: a query that does no work
    trip_pg.append((time.perf_counter() - start) * 1000)
    start = time.perf_counter()
    ai.client.get_document_count()                     # AI Search: the cheapest call it has
    trip_ai.append((time.perf_counter() - start) * 1000)
print(f"trip with no search, middle of 50: PostgreSQL {statistics.median(trip_pg):.1f} ms, AI Search {statistics.median(trip_ai):.1f} ms")

## Step 1. The answer key

**What you will do.** Run the cell. It makes the same 100,000 vectors and 200 questions the trainer loaded, from a
fixed seed, and finds the true top 10 for each question by checking every vector. That is the answer key every store
is scored against.

**What you should see.** 100,000 vectors of 128 numbers, 200 questions, and the time this notebook took to check
every vector for one question.

**Why this step is here.** Recall means nothing without an answer key. In production you build one the same way:
exact search over a sample, once, then score every setting against it.

In [ ]:
bench = make_benchmark_set(n=SETS["128"]["n"], dim=SETS["128"]["dim"])
Q, truth = bench.Q[:100], bench.truth[:100]
start = time.perf_counter()
for q in Q[:20]:
    (bench.X @ q).argsort()[-10:]
print(f"{len(bench.X):,} vectors of {bench.X.shape[1]} numbers, {len(bench.Q)} questions; "
      f"checking every vector here takes {(time.perf_counter() - start) * 1000 / 20:.1f} ms a question")

## Step 2. The dial: recall against speed, one search at a time

**What you will do.** Predict in the `YOUR CALL` cell the lowest `ef` that finds at least 95 percent of the true top
10. `ef` is how many candidates the graph search keeps while it walks: more is slower and finds more. Then run the
sweep. AI Search sets its `ef` in the index definition, not per search (100 here, the lowest it accepts), so it gets
two rows: its graph, and exhaustive (exact) search.

**What you should see.** pgvector: recall 0.736 at ef 10, 0.881 at ef 20, 0.959 at ef 40, 0.970 at ef 80, 1.000 at ef 160, with a middle wait of 1.1 to 1.6 ms and 0.19 to 0.76 ms inside the database; exact search 23 ms. AI Search: recall 1.000 for its graph and for exact, about 20 ms a search, 16 to 17 ms inside the service. pgvector's graph is about 18 times faster for one user at ef 40, and 13 times at equal recall (ef 160); AI Search gives full recall without any tuning.

**Why this step is here.** Recall is a setting you choose, not a property of the store. Every vector store in
production is tuned on this curve.

In [ ]:
# YOUR CALL
PREDICT_EF_FOR_95 = 40

In [ ]:
import matplotlib.pyplot as plt
rows = []
for ef in [10, 20, 40, 80, 160]:
    pg.set(hnsw_ef_search=ef)
    got, ms = one_at_a_time(pg, Q)
    rows.append({"store": "pgvector", "setting": f"ef {ef}", "recall@10": round(recall(truth, got), 3),
                 "p50 ms": round(percentile(ms, 50), 2), "p95 ms": round(percentile(ms, 95), 2),
                 "inside ms": round(median(pg.inside_ms(q) for q in Q[:30]), 2)})
pg.set(enable_seqscan="on", enable_indexscan="off", max_parallel_workers_per_gather=0)    # exact: every row
got, ms = one_at_a_time(pg, Q)
rows.append({"store": "pgvector", "setting": "exact", "recall@10": round(recall(truth, got), 3),
             "p50 ms": round(percentile(ms, 50), 2), "p95 ms": round(percentile(ms, 95), 2),
             "inside ms": round(median(pg.inside_ms(q) for q in Q[:30]), 2)})
pg.set(enable_seqscan="off", enable_indexscan="on", max_parallel_workers_per_gather=2, hnsw_ef_search=40)
for setting, exhaustive in (("graph, ef 100", False), ("exact", True)):
    ai.exhaustive, ai.own_ms = exhaustive, []
    got, ms = one_at_a_time(ai, Q)
    rows.append({"store": "AI Search", "setting": setting, "recall@10": round(recall(truth, got), 3),
                 "p50 ms": round(percentile(ms, 50), 2), "p95 ms": round(percentile(ms, 95), 2),
                 "inside ms": round(median(ai.own_ms), 1)})
ai.exhaustive = False
for row in rows:
    print(f"{row['store']:10} {row['setting']:14} recall {row['recall@10']:.3f}   p50 {row['p50 ms']:7.2f} ms   "
          f"p95 {row['p95 ms']:7.2f} ms   inside the store {row['inside ms']:6.2f} ms")
first_95 = next((r["setting"] for r in rows if r["store"] == "pgvector" and r["recall@10"] >= 0.95), None)
print(f"\nyou predicted ef {PREDICT_EF_FOR_95}; the first pgvector setting at 95 percent: {first_95}")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
for store in ("pgvector", "AI Search"):
    pts = [r for r in rows if r["store"] == store]
    ax.plot([r["p50 ms"] for r in pts], [r["recall@10"] for r in pts], marker="o", label=store)
    for r in pts:
        ax.annotate(r["setting"], (r["p50 ms"], r["recall@10"]), textcoords="offset points", xytext=(4, -9), fontsize=7)
ax.set_xscale("log"); ax.set_xlabel("p50 milliseconds per search (log scale)"); ax.set_ylabel("recall@10")
ax.set_title("Recall against time, one search at a time"); ax.legend(); plt.show()

## Step 3. Many users at once

**What you will do.** Predict which store answers more searches a second when 32 users search at once. Then run the
load test: 400 searches with 1, then 8, then 32 running at the same time, for pgvector at ef 40 and AI Search's graph.

**What you should see.** pgvector: about 590 searches a second for 1 user, 3,330 for 8 and 2,260 for 32 (at least: one notebook could send no more), with the slowest 1 in 100 (p99) at 28 ms. AI Search Basic with one replica: about 60, 95 and 106 searches a second, with p99 at 802 ms for 32 users: it is full at about 100 a second, and the extra users wait in line.

**Why this step is here.** A store that is fast for one user can choke under load. Production teams report searches
a second together with p95 and p99, the waits the slowest users feel, never the average alone.

**Two cautions.** The whole class shares both stores: while other pairs run this step, your numbers drop, so your
trainer may run it once on the projector instead. And one notebook can send only so many searches a second, because
Python runs one thread at a time: when pgvector's number stops rising, the notebook is the limit, not the database. A
real load test spreads the users over several machines.

In [ ]:
# YOUR CALL: "pgvector" or "AI Search"
PREDICT_MORE_AT_32 = "pgvector"

In [ ]:
load = []
for name, store in (("pgvector", pg), ("AI Search", ai)):
    for users in (1, 8, 32):
        row = {"store": name, **under_load(store, list(Q), users)}
        load.append(row)
        print(f"{name:10} {users:2d} users   {row['searches a second']:7.1f} a second   p50 {row['p50 ms']:7.1f} ms   "
              f"p95 {row['p95 ms']:7.1f} ms   p99 {row['p99 ms']:7.1f} ms")
best = max((r for r in load if r["users"] == 32), key=lambda r: r["searches a second"])["store"]
print(f"\nyou predicted {PREDICT_MORE_AT_32}; more searches a second at 32 users: {best}")

## Step 4. A filter: only one label, a tenth of the data

**What you will do.** Run the cell. Thirty questions, each limited to rows with one label. pgvector runs it twice:
plain, then with iterative scan switched on. AI Search filters before it searches.

**What you should see.** pgvector plain: 5.2 rows of 10, recall 0.480. With iterative scan: 10 of 10, recall 0.927. AI Search: 10 of 10, recall 1.000.

**Why this step is here.** Filters are how access rules, dates and tenants reach a vector search (Day 11). A store
that filters after its graph walk returns fewer results with no error. Ask how a store filters before you trust it.

In [ ]:
labels = [int(bench.labels[int(t[0])]) for t in truth[:30]]
keys = [exact_top_k(bench.X, q, allowed=bench.labels == lab) for q, lab in zip(Q[:30], labels)]
for name, store, extra in (("pgvector, plain", pg, {"hnsw_iterative_scan": "off"}),
                           ("pgvector, iterative scan", pg, {"hnsw_iterative_scan": "relaxed_order"}),
                           ("AI Search", ai, {})):
    if extra:
        store.set(**extra)
    got = [store.search(q, label=lab) for q, lab in zip(Q[:30], labels)]
    print(f"{name:26} rows back {np.mean([len(g) for g in got]):4.1f} of 10   recall@10 {recall(keys, got):.3f}")
pg.set(hnsw_iterative_scan="off")

## Step 5. Real-sized vectors: 1,536 numbers

**What you will do.** Run the cell. The same comparison on the second set: 20,000 vectors of 1,536 numbers, the size
text-embedding-3-small produces. It is a smaller set (20,000 rows), so compare recall more than speed.

**What you should see.** pgvector at ef 40: recall 0.800, 1.8 ms a search. AI Search: recall 1.000, 23 ms. Longer vectors need a higher ef for the same recall: compare speeds only at equal recall.

**Why this step is here.** A benchmark on small toy vectors flatters every store. Measure on the size you will ship.

In [ ]:
big = make_benchmark_set(n=SETS["1536"]["n"], dim=SETS["1536"]["dim"])
pg_big = PgStore(PG_CONNECTION, SETS["1536"]["table"]); pg_big.check(SETS["1536"]["n"])
ai_big = SearchStore(SEARCH_ENDPOINT, SEARCH_QUERY_KEY, SETS["1536"]["index"]); ai_big.check(SETS["1536"]["n"])
Qb, truth_b = big.Q[:100], big.truth[:100]
for name, store in (("pgvector, ef 40", pg_big), ("AI Search, graph", ai_big)):
    if store is pg_big:
        store.set(hnsw_ef_search=40)
    got, ms = one_at_a_time(store, Qb)
    inside = median(store.inside_ms(q) for q in Qb[:30]) if store is pg_big else median(store.own_ms)
    print(f"{name:18} recall {recall(truth_b, got):.3f}   p50 {percentile(ms, 50):6.2f} ms   inside the store {inside:6.2f} ms")

## Step 6. What a million searches cost

**What you will do.** Check the two prices in the cell against today's Azure price list, then run it. It divides each
store's price per hour by the searches a second it sustained at 32 users in Step 3.

**What you should see.** pgvector $0.251 an hour at 2,260 or more searches a second: about $0.03 per million searches or less. AI Search Basic $0.133 an hour at about 100 a second: about $0.35 per million.

**Why this step is here.** The fastest store is not always the cheapest per search, and the cheapest per hour is not
always the cheapest per search. This is the number that goes in the decision memo.

In [ ]:
# The searches a second come from Step 3. Where the notebook, not the store, was the limit, this cost is an upper bound.
# US dollars per hour, list price, Central India, November 2026. Check yours on the Azure pricing pages.
PRICE_PER_HOUR = {"pgvector": 0.251,      # Azure Database for PostgreSQL, General Purpose D2ds_v5 (2 vCores), compute only
                  "AI Search": 0.133}     # Azure AI Search, Basic tier, one replica
for name in PRICE_PER_HOUR:
    qps = next(r["searches a second"] for r in load if r["store"] == name and r["users"] == 32)
    per_million = PRICE_PER_HOUR[name] / (qps * 3600) * 1_000_000
    print(f"{name:10} ${PRICE_PER_HOUR[name]:.3f} an hour, {qps:7.1f} searches a second at 32 users: "
          f"${per_million:.3f} per million searches")
print("\nThis assumes the store is busy every second you pay for. A store idle half the day costs twice as much per search.")

## Step 7. Write down what you found

**What you will do.** Run the cell, then paste its lines into the write-up cell and answer the three questions.

**Why this step is here.** A benchmark is only useful as a decision. Write the decision down with the numbers that
support it, and the conditions under which you would change your mind.

In [ ]:
pgvector_40 = next(r for r in rows if r["store"] == "pgvector" and r["setting"] == "ef 40")
print(f"- trip with no search: PostgreSQL {statistics.median(trip_pg):.1f} ms, AI Search {statistics.median(trip_ai):.1f} ms")
print(f"- pgvector at ef 40: recall {pgvector_40['recall@10']}, inside the store {pgvector_40['inside ms']} ms")
print(f"- first pgvector setting at 95 percent recall: {first_95}")
for r in load:
    if r["users"] == 32:
        print(f"- {r['store']} at 32 users: {r['searches a second']} a second, p99 {r['p99 ms']} ms")

## Your write-up

**Pair:**

### 1. Which store would you choose for FreshCart's support search, and why? Use three numbers from this notebook.

### 2. What would make you change your mind? (more data, more users, filters, cost, a team that already runs PostgreSQL)

### 3. One thing this benchmark does not measure that you would test before production.